# Test of DuckDB

- Author: github.com/cyrilby
- Last meaningful update: 13-10-2026

The purpose of this notebook is to simply test working with `DuckDB` rather than `parquet` files to see whether that is feasible.

## Setting up

In [ ]:
import pandas as pd
import duckdb
from storage import get_s3_access, manage_s3_file, connect_to_db

In [2]:
# Importing the credentials for working with object storage
storage_options, bucket = get_s3_access()

## Populating the database with local files

In [4]:
# Downloading the files currently stored as parquet
operation_fmt = pd.read_parquet(
    f"s3://{bucket}/operation_fmt.parquet", storage_options=storage_options
)
station_impact = pd.read_parquet(
    f"s3://{bucket}/station_impact.parquet", storage_options=storage_options
)

mapping_messages = pd.read_pickle(
    f"s3://{bucket}/mapping_messages.pkl", storage_options=storage_options
)
mapping_stations = pd.read_pickle(
    f"s3://{bucket}/mapping_stations.pkl", storage_options=storage_options
)
operation_raw = pd.read_pickle(
    f"s3://{bucket}/operation_raw.pkl", storage_options=storage_options
)

In [ ]:
# Creating a local DuckDB and writing data to it
conn = connect_to_db(read_only=False)
conn.execute("CREATE OR REPLACE TABLE operation_fmt AS SELECT * FROM operation_fmt")
conn.execute("CREATE OR REPLACE TABLE station_impact AS SELECT * FROM station_impact")
conn.execute(
    "CREATE OR REPLACE TABLE mapping_messages AS SELECT * FROM mapping_messages"
)
conn.execute(
    "CREATE OR REPLACE TABLE mapping_stations AS SELECT * FROM mapping_stations"
)
conn.execute("CREATE OR REPLACE TABLE operation_raw AS SELECT * FROM operation_raw")
conn.close()

In [9]:
# Uploading the db to S3 storage
manage_s3_file("database.duckdb", "database.duckdb", "upload")

## Downloading the database and querying it locally

In [11]:
# Downloading the database
manage_s3_file("database.duckdb", "database.duckdb", "download")

In [12]:
# Querying the local database
conn = connect_to_db()
query = """
SELECT *
FROM operation_fmt
LIMIT 1000;
"""
df = conn.execute(query).df()
conn.close()

df.head(5)

,rounded_timestamp,line,timestamp,status_dk,date,day,weekday,weekday_n,day_type,month,...,train_changing,affected_stations,status_disruption,status_en_short,disruption_start,disruption_end,disruption_duration_min,disruption_duration_hours,n_impacted_stations,pct_impacted_stations
0,2024-02-20 15:00:00,M1,2024-02-20 15:00:14,Vi kører efter planen,2024-02-20,20,Tuesday,1,Workdays,February,...,0.0,NaN,False,Normal service,NaT,NaT,NaN,NaN,0,0.0
1,2024-02-20 15:10:00,M1,2024-02-20 15:10:14,Vi kører efter planen,2024-02-20,20,Tuesday,1,Workdays,February,...,0.0,NaN,False,Normal service,NaT,NaT,NaN,NaN,0,0.0
2,2024-02-20 15:20:00,M1,2024-02-20 15:20:14,Vi kører efter planen,2024-02-20,20,Tuesday,1,Workdays,February,...,0.0,NaN,False,Normal service,NaT,NaT,NaN,NaN,0,0.0
3,2024-02-20 15:30:00,M1,2024-02-20 15:30:14,Vi kører efter planen,2024-02-20,20,Tuesday,1,Workdays,February,...,0.0,NaN,False,Normal service,NaT,NaT,NaN,NaN,0,0.0
4,2024-02-20 15:40:00,M1,2024-02-20 15:40:14,Vi kører efter planen,2024-02-20,20,Tuesday,1,Workdays,February,...,0.0,NaN,False,Normal service,NaT,NaT,NaN,NaN,0,0.0


# IDEA

Maybe I can use the `manage_s3_file()` function to upload the latest changes to my mapping table?